# Aggregate by mountain range

The fleet's per-tile partial sums → the mountain-range cube (runoff-onset statistics per GMBA range × 100 m
elevation × 15° aspect × CHILI insolation class × water year), the per-range ERA5-Land anomaly zonal means
merged into it, and the per-range metrics table every other notebook in this folder plots. Run it after the
three GitHub Actions workflows and before the other notebooks here; each section re-runs on its own.

| | |
| --- | --- |
| Reads | `partials/<version>/tile_*.parquet`, downloaded from Azure (`snowmelt_runoff_onset_analysis/partials/<version>/`) when the SAS token is available, otherwise the local cache as is; the GMBA Inventory v2.0 (standard 300) and the USGS continents, read from the web; the ERA5-Land anomaly group of the version's icechunk repository on Azure |
| Writes | `data/aggregation/<version>/all_mountain_ranges_<filter>.nc` (one cube per pixel filter), `data/aggregation/<version>/era5_anomaly_mountain_ranges.nc`, `results/<version>/mountain_range_metrics.csv` (tracked) |
| Needs | the Azure SAS token for the partials download and the ERA5-Land step; without it (the CI smoke test on two fixture tiles) the ERA5 step is skipped and the metrics table has no temperature sensitivity |

What a partials row is, why sums are enough, and what changed against the 2025 workflow: `pipeline/README.md`
and `docs/aggregation_lineage.md`.

In [ ]:
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats as sps

from gsro_analysis import aggregate, era5, paths, results, settings

In [ ]:
config = settings.load_config()          # the dataset version lives in settings.CONFIG_FILE
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'mountain_ranges'
FILTER_TAGS = list(aggregate.FILTERS)    # the pixel filters the fleet emitted: 'full_dataset' and 'fcf_lte_50' (the analyses' rule)
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)   # analyses/mountain_ranges/data/aggregation/<version>/
results_dir = paths.resultsdir(UNIT, VERSION)            # analyses/mountain_ranges/results/<version>/

# the Azure SAS token is needed for the partials download and the ERA5-Land step; without it (the CI smoke test on the
# fixture tiles) the notebook keeps going: using the local partials cache as is, skipping the ERA5-Land section
try:
    config.sas_token
    HAVE_AZURE = True
except ValueError as e:
    HAVE_AZURE = False
    print(f'no Azure SAS token: using the local partials cache as is, skipping the ERA5-Land section ({e})')
print(f'{VERSION} | water years {WATER_YEARS[0]}-{WATER_YEARS[-1]} | filters {FILTER_TAGS} | Azure: {HAVE_AZURE}')
print(f'cubes -> {aggregation_dir}')

## 1. The fleet's partial sums, one parquet per tile

Every row is one tile's contribution to one cell of the cube: the pixels of one (filter, unit type, unit id,
elevation / aspect / latitude bin, CHILI class) with their count and the sums the statistics need
(Σ median, Σ median², per water year Σ onset, Σ onset², Σ anomaly, Σ anomaly², the CHILI and forest-cover
correlation sums). A unit that spans several tiles is several rows; adding them is the reduce.

In [ ]:
partials_dir = paths.partials_cache(VERSION)                        # partials/<version>/ (gitignored)
if HAVE_AZURE:
    partial_files = aggregate.sync_partials(config, partials_dir)   # downloads the tiles missing from the cache, drops stale ones
else:
    partial_files = sorted(partials_dir.glob('tile_*.parquet'))
print(f'{len(partial_files)} tiles in {partials_dir}')

In [ ]:
tile_partials_df = pd.read_parquet(partial_files[0])
print(f'{partial_files[0].name}: {len(tile_partials_df)} rows x {len(tile_partials_df.columns)} columns; '
      'one row = one tile\'s pixels in one (filter, unit type, unit id, bins, CHILI class) cell')
tile_partials_df

In [ ]:
# Sum the partials over tiles, keeping only this unit type. The reduce is a sum over identical keys, so
# summing BATCH tiles at a time gives the same result as concatenating everything first, at a fraction of the
# memory (the full campaign is ~14 M rows; a mountain-range tile alone is ~10 k rows). Each tile is cut down to
# this unit before it joins the batch. min_count=1 keeps a column NaN when no tile reported it.
KEY_COLS = ['filter_tag', 'unit_type', 'unit_id', 'elevation', 'aspect', 'latitude', 'chili_class']
BATCH = 50
t0 = time.time()
summed_partials_df, n_rows = None, 0
for i in range(0, len(partial_files), BATCH):
    tiles = []
    for f in partial_files[i:i + BATCH]:
        tile_df = pd.read_parquet(f)
        if 'unit_type' not in tile_df.columns:    # a verified-empty tile (no pixel with a valid median passed the filters)
            continue
        tiles.append(tile_df[tile_df['unit_type'] == UNIT].drop(columns=['tile_row', 'tile_col'], errors='ignore'))
    batch_df = pd.concat(tiles, ignore_index=True)
    n_rows += len(batch_df)
    batch_sums_df = batch_df.groupby(KEY_COLS, sort=False, dropna=False).sum(min_count=1)
    del tiles, batch_df
    if summed_partials_df is None:
        summed_partials_df = batch_sums_df
    else:
        summed_partials_df = (pd.concat([summed_partials_df, batch_sums_df])
                              .groupby(level=KEY_COLS, sort=False, dropna=False).sum(min_count=1))
summed_partials_df = summed_partials_df.reset_index()
print(f'{n_rows:,} {UNIT} partial rows from {len(partial_files)} tiles summed into {len(summed_partials_df):,} '
      f'cube cells ({time.time() - t0:.0f}s)')
summed_partials_df

## 2. Range names, centroids and continents

The partials key ranges by `GMBA_V2_ID`. The display name is the inventory's `MapName`, or `Level_04` where several
polygons share a `MapName` (the three Andes cordilleras). The centroid is computed in an equal-area projection; a
range's continent is the USGS polygon nearest its centroid, with Australia folded into Oceania.

In [ ]:
# GMBA Inventory v2.0, standard 300 (Snethlage et al. 2022), read straight from EarthEnv
gmba_gdf = gpd.read_file('zip+' + settings.GMBA_URL)
gmba_gdf

In [ ]:
# USGS continents
continents_gdf = gpd.read_file('zip+' + settings.CONTINENTS_URL)
continents_gdf

In [ ]:
range_ids = sorted(summed_partials_df['unit_id'].unique().astype(int))
gmba_by_id_gdf = gmba_gdf.set_index('GMBA_V2_ID').loc[range_ids]
shared_map_name = gmba_by_id_gdf['MapName'].duplicated(keep=False)
range_names = np.where(shared_map_name, gmba_by_id_gdf['Level_04'], gmba_by_id_gdf['MapName'])
centroids = gpd.GeoSeries(gmba_by_id_gdf.geometry.to_crs('EPSG:6933').centroid, crs='EPSG:6933').to_crs('EPSG:4326')
centroid_points_gdf = gpd.GeoDataFrame({'GMBA_V2_ID': range_ids}, geometry=centroids.values, crs='EPSG:4326')
nearest_continent_gdf = gpd.sjoin_nearest(centroid_points_gdf, continents_gdf[['CONTINENT', 'geometry']], how='left')
nearest_continent_gdf = nearest_continent_gdf[~nearest_continent_gdf.index.duplicated()]
range_metadata_df = pd.DataFrame({
    'GMBA_V2_ID': range_ids,
    'name': range_names,
    'centroid_latitude': centroids.y.values,
    'centroid_longitude': centroids.x.values,
    'continent': nearest_continent_gdf['CONTINENT'].replace(aggregate.CONTINENT_MERGE).values,
}).set_index('GMBA_V2_ID')
range_metadata_df

## 3. ERA5-Land anomaly zonal means per range (needs the Azure token)

For every range, water year and hemisphere-aware month, the mean of the ERA5-Land monthly anomaly (each of the 8
variables minus its per-pixel median over all water years) over the 0.1° cells the polygon covers. The weight of a
cell is its coverage fraction × its seasonal-snow fraction × whether the dataset has an onset value there that year
(both masks from the public pyramid), so the climate mean describes the same pixels the onset statistics do.
`era5.zonal_anomalies` does this for all ranges in one streamed pass over the store (~4 minutes).

In [ ]:
# GMBA polygons clipped before the zonal join (multipart / antimeridian or spurious extents), as (minx, miny, maxx, maxy),
# and the ranges skipped entirely
RANGE_GEOMETRY_FIXES = {
    'Aleutian Ranges': (-179.9, 0, -100, 90),
    'Central Range': (120.9, -20, 155, 10),
    'Arctic Ocean': (-179.9, 76, 179.9, 89.9),
    'South Atlantic Islands': (-62, -55, -58, -48),
}
SKIP_RANGES = {'South Atlantic Islands'}   # tiny area, spurious data

era5_zonal_path = aggregation_dir / f'era5_anomaly_{UNIT}.nc'
REBUILD_ERA5_ZONAL = False      # True recomputes even if the file exists (after an ERA5-Land rebuild)
if HAVE_AZURE and (REBUILD_ERA5_ZONAL or not era5_zonal_path.exists()):
    era5_anomaly_ds = era5.open_anomaly(config)     # the anomaly group of the version's ERA5-Land icechunk repository
    era5_zonal_ds = era5.zonal_anomalies(config, gmba_gdf, 'GMBA_V2_ID', geometry_fixes=RANGE_GEOMETRY_FIXES,
                                         skip_names=SKIP_RANGES, anomaly_ds=era5_anomaly_ds)
    era5_zonal_ds.attrs['unit_type'] = UNIT
    era5_zonal_ds.to_netcdf(era5_zonal_path, encoding={v: {'zlib': True, 'complevel': 4} for v in era5_zonal_ds.data_vars})
    print(f'wrote {era5_zonal_path}')
if era5_zonal_path.exists():
    era5_zonal_ds = xr.open_dataset(era5_zonal_path)
else:
    era5_zonal_ds = None
    print(f'no {era5_zonal_path.name}: the cubes get no climate variables (rerun this section with the Azure token)')
era5_zonal_ds

## 4. The cube, one file per pixel filter

`aggregate.reduce_partials` turns the summed rows into bin means (Σx / n), standard deviations
(√(Σx² / n − mean²)), pixel counts and the CHILI / forest-cover correlations on the dense
`mountain_range × elevation × aspect × chili_class × water_year` grid. The ranges are then named, their
centroid and continent attached as coordinates, and the ERA5-Land zonal means merged on
`(mountain_range, water_year, month)`.

In [ ]:
group = UNIT
for filter_tag in FILTER_TAGS:
    t0 = time.time()
    try:
        cube_ds = aggregate.reduce_partials(summed_partials_df, group, filter_tag, WATER_YEARS)
    except ValueError as e:                  # no rows for this filter (a partially processed version)
        print(f'{group}/{filter_tag}: {e}')
        continue
    # names, centroids and continents as coordinates of the mountain_range axis
    ids = cube_ds['mountain_range'].values.astype(int)
    meta_df = range_metadata_df.loc[ids]
    cube_ds = cube_ds.assign_coords(
        GMBA_V2_ID=('mountain_range', ids),
        centroid_latitude=('mountain_range', meta_df['centroid_latitude'].values),
        centroid_longitude=('mountain_range', meta_df['centroid_longitude'].values),
        continent=('mountain_range', meta_df['continent'].values.astype(str)),
    ).assign_coords(mountain_range=meta_df['name'].values.astype(str)).sortby('mountain_range')
    # the ERA5-Land zonal means; a range the zonal join skipped keeps its pixels and gets NaN climate
    if era5_zonal_ds is not None:
        zonal_ds = era5_zonal_ds.reindex(GMBA_V2_ID=cube_ds['GMBA_V2_ID'].values)
        zonal_ds = zonal_ds.rename({'GMBA_V2_ID': 'mountain_range'}).assign_coords(mountain_range=cube_ds['mountain_range'].values)
        zonal_ds = zonal_ds.drop_vars([c for c in zonal_ds.coords if c not in ('mountain_range', 'water_year', 'month')])
        cube_ds = xr.merge([cube_ds, zonal_ds], combine_attrs='drop_conflicts')
    cube_ds.attrs.update({'dataset_version': VERSION, 'n_tiles': len(partial_files),
                          'produced_by': 'analyses/mountain_ranges/0_aggregate_by_mountain_range.ipynb'})
    cube_path = aggregation_dir / f'all_{group}_{filter_tag}.nc'
    encoding = {v: {'zlib': True, 'complevel': 4, **({'dtype': 'float32'} if cube_ds[v].dtype.kind == 'f' else {})}
                for v in cube_ds.data_vars}
    cube_ds.to_netcdf(cube_path.with_suffix('.nc.tmp'), encoding=encoding)
    cube_path.with_suffix('.nc.tmp').replace(cube_path)
    print(f'wrote {cube_path.name}: {cube_path.stat().st_size / 1e6:.1f} MB, dims {dict(cube_ds.sizes)} ({time.time() - t0:.0f}s)')

In [ ]:
mountain_ranges_ds = xr.open_dataset(aggregation_dir / f'all_{UNIT}_fcf_lte_50.nc')
mountain_ranges_ds

In [ ]:
# a first look: the elevation profile of the median runoff onset for the six ranges with the most pixels
pixels_per_range_da = mountain_ranges_ds['runoff_onset_median_n'].sum(['elevation', 'aspect', 'chili_class'])
largest_ranges = pixels_per_range_da.sortby(pixels_per_range_da, ascending=False)['mountain_range'].values[:6]
elevation_profile_da = aggregate.weighted_mean(mountain_ranges_ds.sel(mountain_range=largest_ranges), 'runoff_onset_median', ['aspect', 'chili_class'])
elevation_profile_da.plot.line(x='elevation', hue='mountain_range', marker='.', figsize=(8, 4))
plt.ylabel('median runoff onset [day of water year]')

## 5. The per-range metrics table

One row per range, written to `results/<version>/mountain_range_metrics.csv` with provenance columns: pixel count,
pixel-weighted mean MAD and median onset, two lapse rates, the mean onset anomaly per water year, and the spring
temperature sensitivity. The other notebooks in this folder plot this table; the two composite world maps take
their choropleth from it. Rules, all visible below: bins with fewer than `MIN_PIXELS_PER_BIN` pixels are masked
first; a range-year counts only if at least `MIN_YEAR_FRACTION` of the range's median pixels have data that year;
regressions need at least three points. With at most one point per water year, the Theil-Sen slope is the robust
estimate to quote next to the OLS one.

In [ ]:
MIN_PIXELS_PER_BIN = 100       # a (range, elevation, aspect, CHILI class) bin needs at least this many pixels
MIN_YEAR_FRACTION = 0.1        # a range-year needs this share of the range's median-pixel count to count
SPRING_MONTHS = ['spring_month_1', 'spring_month_2', 'spring_month_3']
BIN_DIMS = ['elevation', 'aspect', 'chili_class']

metrics_ds = aggregate.threshold(mountain_ranges_ds, MIN_PIXELS_PER_BIN)   # bin means (and stds) with fewer pixels -> NaN
metrics_df = pd.DataFrame({
    'name': metrics_ds['mountain_range'].values,
    'GMBA_V2_ID': metrics_ds['GMBA_V2_ID'].values,
    'continent': metrics_ds['continent'].values,
    'centroid_latitude': metrics_ds['centroid_latitude'].values,
    'centroid_longitude': metrics_ds['centroid_longitude'].values,
    'total_pixels_in_range': metrics_ds['runoff_onset_median_n'].sum(BIN_DIMS).values,
    'mean_mad_days': aggregate.weighted_mean(metrics_ds, 'runoff_onset_mad', BIN_DIMS).values,
    'mean_median_onset_dowy': aggregate.weighted_mean(metrics_ds, 'runoff_onset_median', BIN_DIMS).values,
}).set_index('name')
metrics_df

In [ ]:
# Lapse rate 1: sqrt(count)-weighted regression of the bin median onset on elevation over all (elevation, aspect)
# bins with more than MIN_PIXELS_PER_BIN pixels, CHILI classes folded together; needs >= 2 bins spanning >= 100 m
collapsed_ds = aggregate.collapse(metrics_ds)      # the CHILI axis folded exactly (count-weighted means, pooled stds)
rows = []
for name in collapsed_ds['mountain_range'].values:
    range_ds = collapsed_ds.sel(mountain_range=name)
    onset = range_ds['runoff_onset_median'].values
    counts = range_ds['runoff_onset_median_n'].values.astype(float)
    elevation = np.broadcast_to(range_ds['elevation'].values[:, None], onset.shape)
    ok = np.isfinite(onset) & (counts > MIN_PIXELS_PER_BIN)
    lapse_rate, r2 = np.nan, np.nan
    if ok.sum() >= 2 and (elevation[ok].max() - elevation[ok].min()) >= 100:
        weights = np.sqrt(counts[ok])
        slope, intercept = np.polyfit(elevation[ok], onset[ok], 1, w=weights)
        predicted = slope * elevation[ok] + intercept
        weighted_mean = np.average(onset[ok], weights=weights)
        r2 = 1 - np.sum(weights * (onset[ok] - predicted) ** 2) / np.sum(weights * (onset[ok] - weighted_mean) ** 2)
        lapse_rate = slope * 100
    rows.append({'name': name, 'lapse_rate_weighted_bins_per_100m': lapse_rate, 'lapse_rate_weighted_bins_r2': r2})
metrics_df = metrics_df.join(pd.DataFrame(rows).set_index('name'))
metrics_df[['lapse_rate_weighted_bins_per_100m', 'lapse_rate_weighted_bins_r2']].describe()

In [ ]:
# Lapse rate 2: linear regression of the aspect- and CHILI-collapsed elevation profile (the choropleth of the
# polar-triplet world map); needs >= 3 elevation bins with data
elevation_profile_da = aggregate.weighted_mean(metrics_ds, 'runoff_onset_median', ['aspect', 'chili_class'])
rows = []
for name in elevation_profile_da['mountain_range'].values:
    profile = elevation_profile_da.sel(mountain_range=name).values
    ok = np.isfinite(profile)
    row = {'name': name, 'snowmelt_lapse_rate_per_100m': np.nan, 'snowmelt_lapse_rate_corr': np.nan, 'snowmelt_lapse_rate_n': int(ok.sum())}
    if ok.sum() >= 3:
        fit = sps.linregress(elevation_profile_da['elevation'].values[ok], profile[ok])
        row.update(snowmelt_lapse_rate_per_100m=fit.slope * 100, snowmelt_lapse_rate_corr=fit.rvalue)
    rows.append(row)
metrics_df = metrics_df.join(pd.DataFrame(rows).set_index('name'))
metrics_df[['snowmelt_lapse_rate_per_100m', 'snowmelt_lapse_rate_corr', 'snowmelt_lapse_rate_n']].describe()

In [ ]:
# the pixel-weighted mean onset anomaly per range and water year; a range-year with fewer valid pixels than
# MIN_YEAR_FRACTION of the range's median-pixel count is masked
range_mean_anomaly_da = aggregate.weighted_mean(metrics_ds, 'runoff_onset_anomaly', BIN_DIMS)
valid_fraction_da = metrics_ds['runoff_onset_n'].sum(BIN_DIMS) / metrics_ds['runoff_onset_median_n'].sum(BIN_DIMS)
range_mean_anomaly_da = range_mean_anomaly_da.where(valid_fraction_da >= MIN_YEAR_FRACTION)
for year in WATER_YEARS:
    metrics_df[f'runoff_onset_anomaly_WY{year}'] = range_mean_anomaly_da.sel(water_year=year).values
range_mean_anomaly_da

In [ ]:
# Spring-temperature sensitivity: the annual mean onset anomaly regressed on the range's mean spring 2 m temperature
# anomaly — OLS slope [days per degree C], r, p, n — plus the Theil-Sen slope with its 95 % bounds
if 'temperature_2m' in metrics_ds:
    spring_temperature_anomaly_da = metrics_ds['temperature_2m'].sel(month=SPRING_MONTHS).mean('month')
    rows = []
    for name in metrics_ds['mountain_range'].values:
        pairs_df = pd.DataFrame({'x': spring_temperature_anomaly_da.sel(mountain_range=name).values,
                                 'y': range_mean_anomaly_da.sel(mountain_range=name).values}).dropna()
        row = {'name': name, 'anomaly_n': len(pairs_df), 'anomaly_slope': np.nan, 'anomaly_corr': np.nan, 'anomaly_pval': np.nan,
               'theil_sen_slope': np.nan, 'theil_sen_low': np.nan, 'theil_sen_high': np.nan}
        if len(pairs_df) >= 3:
            ols = sps.linregress(pairs_df['x'], pairs_df['y'])
            theil_sen = sps.mstats.theilslopes(pairs_df['y'], pairs_df['x'], 0.95)
            row.update(anomaly_slope=ols.slope, anomaly_corr=ols.rvalue, anomaly_pval=ols.pvalue,
                       theil_sen_slope=theil_sen[0], theil_sen_low=theil_sen[2], theil_sen_high=theil_sen[3])
        rows.append(row)
    metrics_df = metrics_df.join(pd.DataFrame(rows).set_index('name'))
else:
    print('no ERA5-Land variables in the cube: the temperature sensitivity is skipped')
metrics_df

In [ ]:
metrics_path = results_dir / 'mountain_range_metrics.csv'
metrics_df = metrics_df.reset_index().round(3).assign(**results.provenance(config))   # _version, _git_sha, _analysis_git_sha, _written_at
metrics_df.to_csv(metrics_path, index=False)
print(f'{len(metrics_df)} ranges x {len(metrics_df.columns)} columns -> {metrics_path}')
metrics_df.head()